# 물류 두뇌 → Qwen 의사결정 모델 (Colab)

**CONNECT AI LAB × AI CITY BUILDERS** · [CONNECT TYCOON](https://github.com/wonseokjung/connect-tycoon)

웹 학습 실험실에서 내가 알고리즘(Q러닝 · DQN · PPO)과 파라미터(감마 · 보상 · 동전 · 목표 · 꿈 · 호기심)를 정해 키운 **물류 두뇌(brain.json)** 를,
언어 모델 **Qwen3.5-0.8B** 에 옮겨 담아 **물류센터 의사결정 AI 모델**로 만들고 Hugging Face에 올립니다.

1. **두뇌 불러오기** · 내 brain.json (없으면 예시 두뇌)
2. **문제집** · 두뇌가 하루를 운영하는 동안 10분마다 보기 7개를 하나씩 해 보고, 하루 끝까지 다시 돌려 점수를 매깁니다
3. **학습 전 Qwen** · 아무것도 배우지 않은 Qwen을 물류센터장 자리에 앉혀 봅니다
4. **지도학습** · 정답 보기를 따라 배웁니다 (LoRA)
5. **강화학습** · 점수가 높은 보기의 확률을 올립니다
6. **공개** · Hugging Face에 `connectailab-rl-이름-버전-logistics-qwen`

> 메뉴 **런타임 → 런타임 유형 변경 → GPU (T4 이상)**. 왼쪽 🔑(보안 비밀)에 `HF_TOKEN`(Write 권한)을 넣어 두면 마지막 셀에서 바로 올라갑니다.

In [ ]:
# 설치 (2~3분)
!pip -q install "connect-tycoon @ git+https://github.com/wonseokjung/connect-tycoon" "transformers==5.18.0" peft accelerate huggingface_hub
!pip -q install flash-linear-attention || echo "(빠른 계산 모듈 없이 진행 · 조금 느릴 뿐 결과는 같습니다)"
!pip -q uninstall -y torchao   # Colab 기본 torchao(0.10)가 최신 peft와 맞지 않습니다 · 이 학습에는 필요 없습니다
import torch, connect_tycoon as ct
print("CONNECT TYCOON", ct.__version__, "· GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임을 GPU로 바꾸세요)")

## 1 · 내 물류 두뇌 불러오기

[학습 실험실](https://www.ezerai.xyz/models/rl)에서 학습을 마치고 **brain.json 내려받기**를 누른 파일입니다.
- **끌어다 놓기** · Colab 왼쪽 **📁(파일)** 창에 brain.json을 끌어다 놓고 실행 (가장 확실)
- **파일 선택 버튼** · 파일 창에 없으면 이 셀 아래에 [파일 선택] 버튼이 뜹니다
- `USE_MY_BRAIN`을 끄면 예시 두뇌(PPO · 기본값)로 해 봅니다

**왜** · 파이썬 시뮬레이터가 웹과 걸음마다 똑같이 움직여서, 웹에서 본 점수가 여기서도 그대로 나와야 합니다.

✅ **정상이면** · 아래 '시험 24일 하루 점수'가 웹 실험실 결과 화면의 점수와 같습니다 (예시 두뇌 279 · 규칙 사장 86)

In [ ]:
USE_MY_BRAIN = True                                    #@param {type:"boolean"}
UPLOAD_BUTTON = True                                   #@param {type:"boolean"}
from connect_tycoon import labenv as E
from connect_tycoon.lab import LabBrain, example, is_lab, ALGO_NAME

if USE_MY_BRAIN:
    from connect_tycoon.colab import find_brain           # 📁에 있으면 그것, 없으면 [파일 선택] 버튼
    path = find_brain(upload_button=UPLOAD_BUTTON)
    assert is_lab(path), "회사 게임(CONNECT TYCOON)의 brain.json입니다 · 그 두뇌는 'Qwen 강화학습' Colab ①을 쓰세요"
    brain = LabBrain.from_file(path)
else:
    brain = example()
assert brain.spec.get("env") == E.ENV_ID, f"다른 버전 환경의 두뇌입니다 ({brain.spec.get('env')}) · 실험실에서 새로 학습해 내려받으세요"
mine = E.evaluate(brain.policy())
rule = E.evaluate(E.rule_policy)
print(f"두뇌 {brain.name} · {ALGO_NAME.get(brain.algo, brain.algo)}")
print(f"시험 24일 하루 점수 {mine['score']:.1f} (웹 {brain.test.get('score', 0):.1f}) · 제시간 출발 {mine['onTimeRate']:.0%} · 규칙 사장 {rule['score']:.1f}")

## 2 · 문제집 · 보기마다 하루 끝까지 다시 돌려 봅니다 (약 2~4분)

**무엇을** · 두뇌가 하루를 운영하는 동안, 10분마다 보기 7개(대기 · 먼저 온 트럭 도킹 · 급한 트럭 도킹 · 초과근무 · 지게차 집중 · 검수 강화 · 임시 인력)를 하나씩 해 보고, 남은 하루를 두뇌로 끝까지 돌립니다.
보기마다 '그렇게 했으면 남은 하루의 보상이 얼마였나'(서로 다른 미래 8개의 평균)가 점수입니다. 연습 때 보상을 어떻게 바꿨든, 점수는 **공식 보상**으로만 매깁니다.

**정답 고르는 법** · 두뇌의 답이 정답이고, 다른 보기가 **확실히** 더 나을 때(10점 넘게)만 그 보기로 바꿉니다 = 두뇌 + 시뮬레이터의 첨삭.
점수 1등을 그대로 정답으로 쓰면 운 좋게 높게 나온 보기를 고르게 되어 오히려 나빠집니다 (예시 두뇌 279 → 250). 확실할 때만 바꾸면 330 안팎으로 두뇌보다 나아집니다.

**모양** · 상황(사람이 읽는 화면 글) · 질문 · 보기 A~G → 답은 글자 하나

✅ **정상이면** · 30일이면 학습 약 1,800문제 · 검증 240 · 시험 240

In [ ]:
from connect_tycoon.llm import labdata as L, data as D
TRAIN_DAYS = 30                                        # 늘릴수록 문제가 많아집니다 (하루 ≈ 60문제 · 3~5초)
splits = {"train": range(30000, 30000 + TRAIN_DAYS), "valid": range(31000, 31004), "test": range(32000, 32004)}
print(L.build(brain, "lab_data", splits=splits, log=None))
rows = D.load("lab_data", "train")
print(f"두뇌의 답을 그대로 쓴 문제 {sum(r['answer'] == r['teacher'] for r in rows) / len(rows):.0%} · 시뮬레이터가 고친 문제 {sum(r['answer'] != r['teacher'] for r in rows) / len(rows):.0%}")
ex = next(r for r in D.load("lab_data", "test") if r["answer"] != r["teacher"])
print(); print(ex["task"]["state"]); print(ex["task"]["question"])
for o, s in zip(ex["task"]["options"], ex["scores"]): print(f"  {o['label']} {o['description'][:34]:36s} 점수 {s:.1f}")
print("정답", ex["answer"], "· 두뇌의 답", ex["teacher"])

## 3 · 학습 전 Qwen · 물류센터장 자리에 앉혀 보기 (약 2분)

**무엇을** · 아무것도 배우지 않은 Qwen에게 시험 문제를 풀게 하고, 웹 실험실과 똑같은 시험 24일 동안 물류센터를 맡깁니다.
**왜** · 학습 효과를 보려면 출발점을 먼저 재야 합니다.

✅ **정상이면** · 정답률 낮음 · 하루 점수가 규칙 사장 근처이거나 더 낮음

In [ ]:
from connect_tycoon.llm import train_torch as T
test = D.load("lab_data", "test")
base = T.Scorer()
results = {"Qwen 학습 전": D.agreement(test, base.pick)}
results["Qwen 학습 전"].update(L.day_report(base.pick, log=None))
print(results["Qwen 학습 전"])
del base; torch.cuda.empty_cache()

## 4 · 지도학습 · 정답 보기를 따라 배우기 (T4 약 20분)

**무엇을** · 정답 보기 글자를 맞히도록 LoRA를 학습합니다 (모델 전체가 아니라 작은 덧붙임 층만).
**왜** · 두뇌(작은 신경망)가 숫자로 하던 판단을, 언어 모델이 글을 읽고 하도록 옮깁니다.

✅ **정상이면** · 시험 정답률이 크게 오르고, 하루 점수가 두뇌 가까이 올라옵니다

In [ ]:
sft = T.sft("lab_data", "adapters/sft", epochs=1, lr=1e-4, batch=8)
results["지도학습"] = D.agreement(test, sft.pick)
results["지도학습"].update(L.day_report(sft.pick, log=None))
print(results["지도학습"])
del sft; torch.cuda.empty_cache()

## 5 · 강화학습 · 시뮬레이터 점수로 고치기 (T4 약 15분)

**무엇을** · 보기 전체를 한 묶음으로 보고, 점수가 평균보다 높은 보기의 확률은 올리고 낮은 보기는 내립니다. 지도학습 모델에서 너무 멀어지지 않게 붙잡아 둡니다(KL).
**손실** · −Σ p(보기) · 장점(보기) + β · KL(p ‖ 지도학습 모델)

✅ **정상이면** · **후회**(고른 답이 최고점에서 모자란 정도)가 줄어듭니다. 하루 점수는 비슷하거나 조금 오릅니다

In [ ]:
rlm, hist = T.rl("lab_data", "adapters/sft", "adapters/rl", steps=100, batch=8, lr=5e-6, beta=0.2, max_rows=1500)
results["강화학습"] = D.agreement(test, rlm.pick)
results["강화학습"].update(L.day_report(rlm.pick, log=None))
print(f"{'':12s} 정답률  후회   하루 점수  제시간")
for k, v in results.items(): print(f"{k:12s} {v['accuracy']:.1%}  {v['regret']:.2f}  {v['day_score']:7.1f}   {v['on_time']:.0%}")
print(f"{'두뇌':12s}   —      —    {mine['score']:7.1f}   {mine['onTimeRate']:.0%}")
print(f"{'규칙 사장':12s}   —      —    {rule['score']:7.1f}   {rule['onTimeRate']:.0%}")

## 6 · Hugging Face에 올리기

1. https://huggingface.co/settings/tokens 에서 **Write** 토큰을 만듭니다.
2. Colab 왼쪽 🔑(보안 비밀)에 이름 `HF_TOKEN`, 값에 토큰을 넣고 **노트북 액세스**를 켭니다.
3. **이름**을 영문으로 적습니다. 업로드 이름은 `connectailab-rl-이름-버전-logistics-qwen` 입니다 (예: `connectailab-rl-jay-v1-logistics-qwen`).
4. **나를 알리는 글**을 자유롭게 적으면 모델 카드 맨 위 '만든 사람'에 들어갑니다.
5. `PUSH_BRAIN`을 켜면 두뇌(brain.json)도 `connectailab-rl-이름-버전-logistics`로 함께 올라갑니다.

토큰은 코드에 적지 않습니다.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi
from connect_tycoon.llm.hub import make_author, push, repo_name
import json

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
assert HF_TOKEN, "왼쪽 🔑에 HF_TOKEN(Write 권한)을 넣고 노트북 액세스를 켜 주세요"
me = HfApi(token=HF_TOKEN).whoami()["name"]

NAME = "jay"                                         #@param {type:"string"}
VERSION = "v1"                                       #@param {type:"string"}
PRIVATE = False                                      #@param {type:"boolean"}
PUSH_BRAIN = True                                    #@param {type:"boolean"}
MODEL_NAME = repo_name(NAME, VERSION, "logistics-qwen")   # connectailab-rl-이름-버전-logistics-qwen

# 나를 알리는 글 · 형식 없이 자유롭게 (모델 카드 맨 위 '만든 사람'에 그대로 들어갑니다)
ABOUT_ME = '''
'''
author = make_author(ABOUT_ME)

meta = json.load(open("lab_data/meta.json"))
url = push("adapters/rl", repo=f"{me}/{MODEL_NAME}", results=results, meta=meta, backend="torch", token=HF_TOKEN, private=PRIVATE, author=author)
print("올라갔습니다 ·", url)
if PUSH_BRAIN:
    from connect_tycoon.hub import push_lab_brain
    json.dump(brain.spec, open("brain.json", "w"), ensure_ascii=False)
    print("두뇌 ·", push_lab_brain("brain.json", name=NAME, version=VERSION, author=author, token=HF_TOKEN))